In [1]:
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
import numpy as np

In [2]:
# df = pd.read_pickle('data/df.pkl')
# df_2025 = pd.read_pickle('data/df_2025.pkl')

# non_features = [col for col in df.columns if 'ndvi' in col] + ['year','plot_id', 'geometry', 'key_0', 'area_m2']
# features = [col for col in df.columns if col not in non_features]
# target = df['ndvi_integral']
# target_2025 = df_2025['ndvi_integral']

# X = df[features]
# X_2025 = df_2025[features]

# selector = VarianceThreshold(threshold=0.01)
# df_selected= selector.fit_transform(df[features])
# selector = VarianceThreshold(threshold=0.01)

# X_var = selector.fit_transform(X)
# X_var_2025 = selector.transform(X_2025)

# # Keep feature names
# kept_features = X.columns[selector.get_support(indices=True)]
# X_clean = pd.DataFrame(X_var, columns=kept_features, index=X.index)

# # Keep feature names
# kept_features_2025 = X_2025.columns[selector.get_support(indices=True)]
# X_clean_2025 = pd.DataFrame(X_var_2025, columns=kept_features_2025, index=X_2025.index)

# print(f'Original features: {X.shape[1]}')
# print(f'Kept features: {X_clean.shape[1]}')
# print('Removed features:', set(features) - set(kept_features)) 

# print(f'Original features: {X_2025.shape[1]}')
# print(f'Kept features: {X_clean_2025.shape[1]}')
# print('Removed features:', set(features) - set(kept_features_2025)) 

# corr_matrix = X_clean.corr().abs()
# upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
# to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
# X_clean = X_clean.drop(columns=to_drop)
# print('Dropped for correlation:', to_drop)

# corr_matrix_2025 = X_clean_2025.corr().abs()
# upper = corr_matrix.where(np.triu(np.ones(corr_matrix_2025.shape), k=1).astype(bool))
# to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
# X_clean_2025 = X_clean_2025.drop(columns=to_drop)
# print('Dropped for correlation:', to_drop)


# X_clean = X_clean.clip(lower=X_clean.quantile(0.01), upper=X_clean.quantile(0.99), axis=1)

In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold

# ---------------------------
# 1. Load data
# ---------------------------
# Load historical data (2016–2024)
df = pd.read_pickle('data/df.pkl')
# df = df[df['year'] >= 2016]

weights_by_year = {
    2016: 0.5,
    2017: 0.6,
    2018: 0.7,
    2019: 0.8,
    2020: 0.9,
    2021: 1.0,
    2022: 1.1,
    2023: 1.2,
    2024: 1.3
}

# Load holdout year (2025)
df_2025 = df[df['year'] == 2025].copy()
df = df.drop(df_2025.index)

# ---------------------------
# 2. Define features vs. non-features
# ---------------------------
# Non-features: NDVI target variables + metadata columns
non_features = [col for col in df.columns if 'ndvi' in col] \
             + ['year', 'plot_id', 'geometry', 'key_0', 'area_m2']

# Features = everything else
features = [col for col in df.columns if col not in non_features]


# ---------------------------
# 3. Separate target variable
# ---------------------------
# Historical target (2016–2024)
target = df['ndvi_integral']

# 2025 target (for testing only, not training)
target_2025 = df_2025['ndvi_integral']


# ---------------------------
# 4. Extract features for train vs. test
# ---------------------------
X = df[features]         # all historical years
X_2025 = df_2025[features]   # 2025 holdout year


# ---------------------------
# 5. Separate historical years (NEW)
# ---------------------------
# 👉 This is the key addition:
#    We'll split historical df into per-year dictionaries,
#    so you can easily train/test by year (forecast-aware training).
dfs_by_year = {year: df[df['year'] == year] for year in df['year'].unique()}

# Extract feature/target per year if needed later
X_by_year = {year: d[features] for year, d in dfs_by_year.items()}
y_by_year = {year: d['ndvi_integral'] for year, d in dfs_by_year.items()}


# ---------------------------
# 6. Variance threshold filtering
# ---------------------------
# Initialize selector
selector = VarianceThreshold(threshold=0.01)

# Fit on historical data
X_var = selector.fit_transform(X)

# Apply transform to 2025 data
X_var_2025 = selector.transform(X_2025)


# ---------------------------
# 7. Keep feature names
# ---------------------------
kept_features = X.columns[selector.get_support(indices=True)]
X_clean = pd.DataFrame(X_var, columns=kept_features, index=X.index)

kept_features_2025 = X_2025.columns[selector.get_support(indices=True)]
X_clean_2025 = pd.DataFrame(X_var_2025, columns=kept_features_2025, index=X_2025.index)



# ---------------------------
# 8. Print diagnostic info
# ---------------------------
print(f'Original features: {X.shape[1]}')
print(f'Kept features: {X_clean.shape[1]}')
print('Removed features:', set(features) - set(kept_features)) 

print(f'Original features (2025): {X_2025.shape[1]}')
print(f'Kept features (2025): {X_clean_2025.shape[1]}')
print('Removed features (2025):', set(features) - set(kept_features_2025)) 


# ---------------------------
# 9. Correlation filtering
# ---------------------------
# Historical
corr_matrix = X_clean.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
X_clean = X_clean.drop(columns=to_drop)
print('Dropped for correlation (historical):', to_drop)

# 2025
corr_matrix_2025 = X_clean_2025.corr().abs()
upper_2025 = corr_matrix_2025.where(np.triu(np.ones(corr_matrix_2025.shape), k=1).astype(bool))
to_drop_2025 = [column for column in upper_2025.columns if any(upper_2025[column] > 0.9)]
X_clean_2025 = X_clean_2025.drop(columns=to_drop)
print('Dropped for correlation (2025):', to_drop)


# ---------------------------
# 10. Clip extreme values (winsorization)
# ---------------------------
X_clean = X_clean.clip(lower=X_clean.quantile(0.01), 
                       upper=X_clean.quantile(0.99), axis=1)


Original features: 107
Kept features: 107
Removed features: set()
Original features (2025): 107
Kept features (2025): 107
Removed features (2025): set()
Dropped for correlation (historical): ['elev_max', 'elev_mean', 'elev_dev_min', 'elev_dev_max', 'elev_dev_mean', 'ppt_var', 'tmean_avg', 'tmax_max', 'tmax_min', 'tmax_mean', 'tmax_var', 'tmin_max', 'tmin_min', 'tmin_avg', 'vpdmax_mean', 'vpdmax_max', 'vpdmax_min', 'vpdmax_var', 'vpdmin_mean', 'vpdmin_max', 'vpdmin_var', 'gdd_sum', 'tmax_mid_mean', 'tmin_early_max', 'tmean_early_mean', 'tmean_mid_mean', 'tmean_late_mean', 'tmean_mid_min', 'tmean_late_min', 'tmean_mid_max', 'tmean_late_max', 'vpdmax_mid_mean', 'vpdmax_late_mean', 'vpdmax_mid_max', 'vpdmax_late_max', 'vpdmin_early_min', 'vpdmin_late_min', 'vpdmin_early_max', 'vpdmin_mid_max', 'slope_x', 'slope_squared', 'slope_log', 'local_relief', 'vpdmin_mean2', 'vpdmin_max2', 'vpdmin_min2', 'vpdmax_mean2', 'diurnal_temp_range2', 'total_relief_log', 'vpdmax_tmax_min']
Dropped for correl

In [4]:
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [5]:
sample_weights = df['year'].map(weights_by_year).loc[X_clean.index].values

# ---------------------------
# Split 2016–2024 into train + validation
# ---------------------------
# First split off the test set (say 20%)
# X_temp, X_test, y_temp, y_test = train_test_split(
#     X_clean, target, test_size=0.2, random_state=42
# )

# # Then split the remaining into train + validation (say 20% of remaining)
# X_train, X_val, y_train, y_val = train_test_split(
#     X_temp, y_temp, test_size=0.25, random_state=42
# )


X_train, X_val, y_train, y_val = train_test_split(
    X_clean,          # all historical features
    target,           # all historical target
    test_size=0.3,    # 20% validation from 2016–2024
    random_state=42,
    shuffle=True      # shuffle inside 2016–2024 is fine
)
# Split weights **in the same order**
w_train, w_val = train_test_split(
    sample_weights,
    test_size=0.3,
    random_state=42,
    shuffle=True
)
# ---------------------------
# Hold out 2025 as final test set
# ---------------------------
X_test = X_clean_2025
y_test = target_2025


In [6]:
model = XGBRegressor(
    n_estimators=5000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.85,
    colsample_bytree=0.70,
    min_child_weight=7,
    random_state=42,
    eval_metric='rmse',
    reg_alpha = 0.05,
    reg_lambda = 1.2
)



# Map dictionary to each row in X_clean
sample_weights = df['year'].map(weights_by_year).values
sample_weights = sample_weights / sample_weights.mean()

# Then pass this to XGBoost
# model.fit(X_clean, target, sample_weight=sample_weights)


model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    early_stopping_rounds=100,
    sample_weight = w_train,
    verbose=False
)

/home/simonhans/anaconda3/lib/python3.7/site-packages/xgboost/sklearn.py:797: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  UserWarning,


KeyboardInterrupt: 

In [ ]:
y_hat = (model.predict(X_test))
resid = y_hat - y_val
r2 = r2_score(y_val, y_hat)
print(r2)

In [ ]:
len(X_test.columns)

In [ ]:
y_hat_2025 = model.predict(X_test)

resid_2025 = y_hat_2025 - y_test
r2_2025 = r2_score(y_test, y_hat_2025)
print(r2_2025)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# y_test = np.exp(y_test)
# y = np.exp(y)
y_pred = (model.predict(X_clean_2025))

r2 = r2_score(target_2025, y_pred)
print('Test R²:', r2)





plt.figure(figsize=(10, 8))
plt.scatter(target_2025, y_pred, alpha=0.6)
plt.plot([target_2025.min(), target_2025.max()], [target_2025.min(), target_2025.max()], 'r--', linewidth=2)
plt.xlabel('Observed NDVI Integral', fontsize = 16)
plt.ylabel('Predicted NDVI Integral', fontsize = 16)
plt.title('XGBoost Predictions vs Observed', fontsize = 18)
plt.savefig('img/pred_vs_obs.png')
plt.xticks(size = 14)
plt.yticks(size = 14)
plt.show()





plt.figure(figsize=(10, 8))
plt.scatter(target_2025, (y_pred - target_2025)/target_2025 * 100, alpha=0.6)
plt.plot([target_2025.min(), target_2025.max()], [0,0], 'r--', linewidth=2)
plt.xlabel('Observed NDVI Integral', fontsize = 16)
plt.ylabel('Percent Residual', fontsize = 16)
plt.title('XGBoost Predictions vs Observed', fontsize = 18)
plt.savefig('img/pred_vs_obs.png')
plt.xticks(size = 14)
plt.yticks(size = 14)
plt.show()




In [ ]:
X_train.columns

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

years = sorted(X_by_year.keys())  # e.g. [2016, ..., 2024]
results = []

for i in range(len(years) - 1):  
    train_years = years[:i+1]      # train up through this year
    val_year = years[i+1]          # validate on the next year
    
    # Combine training years
    X_train = pd.concat([X_by_year[y] for y in train_years])
    y_train = pd.concat([y_by_year[y] for y in train_years])
    
    # Validation set
    X_val = X_by_year[val_year]
    y_val = y_by_year[val_year]

    # Train model
    model = xgb.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    model.fit(X_train, y_train)

    # Validate
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    results.append({
        "train_years": train_years,
        "val_year": val_year,
        "rmse": rmse
    })

results_df = pd.DataFrame(results)
print(results_df)
